In [ ]:
import os

In [ ]:
%pwd

In [ ]:
os.chdir('..')

In [ ]:
%pwd

In [ ]:
import yaml
from src.dataset import NDBUfesOrganizer, NDBUfesDataset
from src.pipeline import Pipeline
from src.validate_input.params import Parameters

with open("params.yaml", "r") as f:
        params_dict= yaml.load(f, yaml.Loader)

params_model = Parameters.model_validate(params_dict)

In [ ]:
params_model.project

In [ ]:
from src.validate_input.params import ProjectEnum
params_model.project =  ProjectEnum.dys_bin

In [ ]:
params_model.project

In [ ]:
params = params_model

In [ ]:
data_organizer = NDBUfesOrganizer(task=params_model.project.value,
                                      node_type=params_model.node_type,
                                      **params_model.dataset.model_dump())

In [ ]:
batch_size = params.hyperparameters.other.batch_size
(test_paths, test_labels) = data_organizer.data_per_fold(fold=0, train=False)

In [ ]:
import mlflow
from torch.utils.data import DataLoader

In [ ]:
params.hyperparameters.model

In [ ]:
from src.validate_input.params import ModelEnum, Model
params.hyperparameters.model = Model(name=ModelEnum.densenet121)

In [ ]:
train_run_ids = [
    ("84f607f84a6e4144a130092891aa6f6b", "models/project_dys_bin_run_densenet121_sgd_2025_08_04_00_15_21/fold_1/model_densenet121_optim_sgd_best.pt"),
    ("be7574ce664b447191a5948d73fcc039", "models/project_dys_bin_run_densenet121_sgd_2025_08_04_00_15_21/project_dys_bin_run_densenet121_sgd_2025_08_04_00_15_21/fold_2/model_densenet121_optim_sgd_best.pt"),
    ("ff3333a5a9494d1cb216d9dee058a773", "models/project_dys_bin_run_densenet121_sgd_2025_08_04_00_15_21/project_dys_bin_run_densenet121_sgd_2025_08_04_00_15_21/project_dys_bin_run_densenet121_sgd_2025_08_04_00_15_21/fold_3/model_densenet121_optim_sgd_best.pt"),
    ("ac9f19b55734452e966d209cd4716736", "models/project_dys_bin_run_densenet121_sgd_2025_08_04_00_15_21/project_dys_bin_run_densenet121_sgd_2025_08_04_00_15_21/project_dys_bin_run_densenet121_sgd_2025_08_04_00_15_21/project_dys_bin_run_densenet121_sgd_2025_08_04_00_15_21/fold_4/model_densenet121_optim_sgd_best.pt"),
    ("e319473e359846c395bba7cdfb6c5d99","models/project_dys_bin_run_densenet121_sgd_2025_08_04_00_15_21/project_dys_bin_run_densenet121_sgd_2025_08_04_00_15_21/project_dys_bin_run_densenet121_sgd_2025_08_04_00_15_21/project_dys_bin_run_densenet121_sgd_2025_08_04_00_15_21/project_dys_bin_run_densenet121_sgd_2025_08_04_00_15_21/fold_5/model_densenet121_optim_sgd_best.pt")
]

In [ ]:
from src.pipeline.job import TestJob

In [ ]:
mlflow.set_tracking_uri(uri="http://localhost:8000/")

In [ ]:
for (run_id, path) in train_run_ids:
    print(run_id)
    # try:
    #     run_mf = mlflow.search_runs(filter_string=f"attributes.run_id = '{run_id}'", search_all_experiments=True)
    #     path_to_artifact = run_mf["artifact_uri"][0] + "/model"
    #     best_model_path = mlflow.artifacts.list_artifacts((path_to_artifact))[0]
    #     best_model_path = f"{run_mf["artifact_uri"][0]}/{best_model_path.path}"
    #     print(f"Found run_id = {run_id}")
    # except Exception as e:
    #     print(f"Run id = {run_id} was not found.")
    #     print(f"Error found: {e}")
    #     continue
    
    # try:
    #     print(f"Downloading model {best_model_path}")
    #     best_model_local_path = mlflow.artifacts.download_artifacts(best_model_path, dst_path="best_model")
    # except Exception as e:
    #     print(f"Unnabled to download model.")
    #     continue

    best_model_local_path = path
    test_job = TestJob(params, 
                        data_organizer.train_num_classes,
                        best_model_local_path)

    test_paths, test_labels = data_organizer.data_per_fold(fold=0, train=False)
    test_dataset = NDBUfesDataset(test_paths, test_labels, 
                            classes_dict=data_organizer.train_classes_dict,
                            aug=None,
                            transform=data_organizer.test_transform)
    test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True)

    metrics, true_pred_df = test_job.run(test_dataloader, create_artifacts=False)
    metrics = {f"test_{key}": item for key, item in metrics.items()}

    mlflow.log_metrics(metrics, run_id=run_id)
    
    true_pred_df.columns = [f"test_{name}" for name in true_pred_df.columns]
    temp_csv_path = f"true_pred_table_{run_id}.csv"
    true_pred_df.to_csv(temp_csv_path, index=False)
    mlflow.log_artifact(temp_csv_path, artifact_path="true_pred_table", run_id=run_id)

    if os.path.exists(best_model_local_path) or os.path.exists(temp_csv_path):
        # os.remove(best_model_local_path)
        os.remove(temp_csv_path)

In [ ]:
%pip install ipywidgets